In [1]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import torch
from datasets import load_dataset
from tqdm import tqdm

dataset = load_dataset(
    "mikronai/svg-svgrepo", split="train+valid+test", revision="3503985b648ed243f34e00587ac7af7c6da4fedd"
)
dataset = list(dataset)
print("Dataset size:", len(dataset))

dataset = dataset[:1000]  # For testing purposes

from transformers import AutoImageProcessor, AutoModel, CLIPModel, CLIPProcessor

# CLIP
processor_clip = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
model_clip = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to("cuda")

# DINOv2
processor_dino = AutoImageProcessor.from_pretrained("facebook/dinov2-base", use_fast=False)
model_dino = AutoModel.from_pretrained("facebook/dinov2-base").to("cuda")

from itertools import batched

from svgai.img import center_pad_image
from svgai.svg import render_fit

with torch.no_grad():
    for batch in tqdm(batched(dataset, 32), total=len(dataset) // 32):
        rendered_images = []

        # Render svgs to raster images
        for item in batch:
            svg = item["item_svg"]
            rendered_image = render_fit(svg, 512, 512)
            rendered_image = center_pad_image(rendered_image, 512, 512)
            rendered_images.append(rendered_image)

        # CLIP
        inputs = processor_clip(images=rendered_images, return_tensors="pt")
        inputs["pixel_values"] = inputs["pixel_values"].to("cuda")
        embedding = model_clip.get_image_features(**inputs)
        embedding = embedding.cpu().numpy()
        for i, item in enumerate(batch):
            item["embedding_clip"] = embedding[i]

        # DINOv2
        inputs = processor_dino(images=rendered_images, return_tensors="pt")
        inputs["pixel_values"] = inputs["pixel_values"].to("cuda")
        outputs = model_dino(**inputs)
        embedding = outputs.last_hidden_state
        embedding = embedding[:, 0, :].squeeze(1)
        embedding = embedding.cpu().numpy()
        for i, item in enumerate(batch):
            item["embedding_dino"] = embedding[i]

# Create dataset and save to disk
# from datasets import Dataset

# new_dataset = Dataset.from_list(dataset)
# new_dataset.save_to_disk("/var/tmp/xkuchar/svgrepo-precomputed")


Dataset size: 215632


32it [00:27,  1.18it/s]                        


In [5]:
dataset[10]['embedding_clip']

array([ 2.03506351e-02, -3.86350632e-01,  3.32704514e-01, -4.55065578e-01,
       -6.66216075e-01, -6.68378770e-01,  9.24837589e-01,  3.71484160e-02,
        2.60316730e-02, -4.39751983e-01, -1.00584352e+00, -4.01485473e-01,
        3.94997090e-01, -2.38326043e-01,  6.08796030e-02,  2.51929671e-01,
        1.65766999e-02,  4.71967399e-01,  1.89960897e-01,  1.72581762e-01,
        1.24304861e-01,  3.79983693e-01,  3.66847575e-01, -1.13444829e+00,
       -8.53096306e-01, -4.79223058e-02, -6.67406142e-01, -3.70485485e-02,
        1.74282402e-01, -3.35176438e-01, -1.14494789e+00,  3.28147680e-01,
        1.72550038e-01, -4.61284161e-01,  5.36376238e-03,  9.95850265e-02,
       -1.91942751e-02,  1.16534628e-01,  4.70245272e-01, -1.65251911e-01,
       -1.02917027e+00,  2.00157836e-01, -9.02767897e-01, -1.56429455e-01,
       -9.39998552e-02, -6.18593618e-02,  3.91661286e-01, -4.50694531e-01,
       -3.02093238e-01,  9.69069719e-01, -1.49066150e-01,  3.77407759e-01,
        1.62350848e-01, -